## Setup

In [5]:
# To support both python 2 and python 3
from __future__ import division, print_function, unicode_literals

# Common imports
import numpy as np
import os
import pandas as pd


pd.set_option("display.max_rows", 999)

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12


# Ignore useless warnings (see SciPy issue #5998)
import warnings
warnings.filterwarnings(action="ignore", message="^internal gelsd")

In [11]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("California_Houses (1).csv")

df = pd.read_csv(DATA_PATH)

print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'California_Houses (1).csv'

The \<os> module to change your current (default) working directory to PATH, your new working directory

In [ ]:
import os
os.getcwd()
os.chdir(PATH)

**Importing the \<California_Houses.csv> dataset**

In [ ]:
df = pd.read_csv("California_Houses (1).csv")
print(df.head)

**Each row of the dataset represents one district in California**<br>
Have a look a the first and last few rows

In [ ]:
print(df.head())
print(df.tail())

### Creating a categorical  variable \<Closest_city> indicating the closest CA city and drop the distance to each city

In [ ]:
#Preparing the columns to explore
city_columns = {
    "Distance_to_LA": "Los Angeles",
    "Distance_to_SanDiego": "San Diego",
    "Distance_to_SanJose": "San Jose",
    "Distance_to_SanFrancisco": "San Francisco"
}

#Creating the new "Closest city" column
df["Closest_city"] = df[list(city_columns.keys())].idxmin(axis=1).map(city_columns)

df["Closest_city"] = df["Closest_city"].astype("category")

#Dropping the distance's columns
df = df.drop(columns=list(city_columns.keys()))

#Implementing it to the housing df
housing = df

In [ ]:
#Checking the new column
print(df.tail())
print(df.sample(10))
print(df["Closest_city"].value_counts())


**Displaying the summary of your new dataframe**

In [7]:
housing.info()

NameError: name 'housing' is not defined

# Part 1 - Data explorations

### Finding out what categories exist in 'Closest_city' column and how many districts belong to each category.


In [ ]:
housing.groupby('Closest_city').size().reset_index(name='District_count')

### Showing a summary of the quantitative attributes


In [ ]:
cols_to_exclude = ['Latitude', 'Longitude']
columns_to_show = housing.drop(columns=cols_to_exclude)
columns_to_show.describe(include='number')

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
columns_to_show.hist(figsize=(12, 10))
plt.tight_layout()
columns_to_show.plot(kind='box', subplots=True, layout=(3,3), figsize=(12,10))
plt.tight_layout()
plt.show()
plt.show()


In [ ]:
housing["Median_Income"].hist(bins=50)
plt.show()

In [8]:
housing["Median_Income"].plot(kind='box', figsize=(6,6))
plt.show()


NameError: name 'housing' is not defined

In [ ]:
cat=[np.min(housing["Median_Income"])]
for i in [0.20, 0.40, 0.60, 0.80]:
    cat.append(housing["Median_Income"].quantile(i))
cat.append(np.max(housing["Median_Income"]))
print(cat)

In [ ]:
housing["income_cat"]=pd.cut(housing["Median_Income"], bins=cat, labels = [1,2,3,4,5], include_lowest=True)

In [ ]:
housing["income_cat"].value_counts()

In [ ]:
housing["income_cat"].hist();

In [ ]:
housing.info()

### Creating a Test Set through stratified random sampling on the income variable

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
#Creation of a stratified Test set
train_strat, test_strat = train_test_split(
    housing,
    test_size=0.2,
    random_state=42,
    stratify=housing['income_cat']
)

print(test_strat['income_cat'].value_counts(normalize=True))
print(train_strat['income_cat'].value_counts(normalize=True))

Now generating an equivalent random split without stratification

In [ ]:
#Creation of a non stratified Test set
train_random, test_random = train_test_split(
    housing,
    test_size=0.2,
    random_state=42
)

print(test_random['income_cat'].value_counts(normalize=True))
print(train_random['income_cat'].value_counts(normalize=True))

In [9]:
def income_cat_proportions(data):
    return data["income_cat"].value_counts() / len(data)

train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

compare_props = pd.DataFrame({
    "Overall": income_cat_proportions(housing),
    "Stratified": income_cat_proportions(test_strat),
    "Random": income_cat_proportions(test_random),
}).sort_index()

compare_props["Rand. %error"] = 100 * compare_props["Random"] / compare_props["Overall"] - 100
compare_props["Strat. %error"] = 100 * compare_props["Stratified"] / compare_props["Overall"] - 100

NameError: name 'train_test_split' is not defined

In [ ]:
compare_props

**For safety, copying the stratified train set to be used for modeling** <br>


In [ ]:
houses_df = train_strat.copy()

### Data Visualization : scatter plots

**Ploting each row (observation) in the dataset as a geographical point** <br>

In [ ]:
import matplotlib.pyplot as plt

houses_df.plot(
    kind="scatter",
    x="Longitude",
    y="Latitude",
    figsize=(10,10),
    alpha=0.2
)

plt.show()

### Geographic map of California houses values per district with population density

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.image as mpimg
california_img=mpimg.imread("california (1).png")
ax = houses_df.plot(kind="scatter", x="Longitude", y="Latitude", figsize=(14,10),
                       s=houses_df['Population']/100, label="Population",
                       c="Median_House_Value", cmap=plt.get_cmap("jet"),colorbar=False, alpha=0.4)
plt.imshow(california_img, extent=[-124.55, -113.80, 32.45, 42.05], alpha=0.5,cmap=plt.get_cmap("jet"))
plt.ylabel("Latitude", fontsize=14)
plt.xlabel("Longitude", fontsize=14)

prices = houses_df["Median_House_Value"]
tick_values = np.linspace(prices.min(), prices.max(), 11)
cbar = plt.colorbar()
cbar.ax.set_yticklabels(["$%dk"%(round(v/1000)) for v in tick_values], fontsize=14)
cbar.set_label('Median House Value', fontsize=16)

plt.legend(fontsize=16)
plt.show()

### Bivariate Analysis

Computing the correlation matrix of all the quantitative variables <br>

In [10]:
import seaborn as sns
import matplotlib.pyplot as plt

houses_cor = houses_df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(houses_cor, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation matrix")
plt.show()

NameError: name 'houses_df' is not defined

Displaying the most important correlations with the target variable : \<Median_House_Value> <br>


In [ ]:
houses_cor["Median_House_Value"].sort_values(ascending=False)

**Providing the scatter plots of those predictors with the target**

In [ ]:
#Scatter plots of predictors with the target
predictors = [
    'Median_Income',
    'Tot_Rooms',
    'Median_House_Value',
    'Population',
    'Latitude',
    'Longitude',
    'Tot_Bedrooms',
    'Distance_to_coast',
    'Households'
]

fig, axes = plt.subplots(3, 3, figsize=(15,10))  # 2 lines, 3 columns
axes = axes.flatten()

for i, col in enumerate(predictors):
    axes[i].scatter(df[col], df['Median_House_Value'])
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

**Creating three new variables :**
1. Rooms per household
2. Bedrooms per rooms
3. People per household

In [ ]:
#####################################

df['Rooms_per_household'] = df['Tot_Rooms'] / df['Households']
df['Bedrooms_per_room'] = df['Tot_Bedrooms'] / df['Tot_Rooms']
df['People_per_household'] = df['Population'] / df['Households']
print(df.head())

#####################################

**Let us again look at the correlation between the predictors and the target**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

# Part 2 - Preparing data for Machine Learning

## 2.1 - Missing Values

### 2.1.1 - Case when there are missing values in one variable

Let us create a dataset where there are 10% of missing values in one variable

In [ ]:
## Randomly select 10% of the dataset indices to simulate missing values
## random.seed(42) ensures reproducibility of the random selection
## 1651 ≈ 10% of the training set size

import random
random.seed(42)
miss = np.random.choice(houses_df.index, 1651)

In [ ]:
miss

In [ ]:
## Create a copy of the original training set to avoid modifying it directly.
## Working on a copy preserves houses_df intact for future use.

houses_miss = houses_df.copy()

In [ ]:
## Replace the values of Tot_Bedrooms with None (missing values) for the 1651 randomly selected indices stored in "miss".
## This artificially simulates a real-world scenario where ~10% of data is missing.

houses_miss.loc[miss,"Tot_Bedrooms"] = None

In [ ]:
#To get the info oh the modified copy training test
houses_miss.info()

In [ ]:
#We drop the rows associated to the missing values
houses_drop = houses_miss.dropna(subset=["Tot_Bedrooms"])

In [ ]:
#We check
houses_drop.info()

In [ ]:
## Your comment here ##
#We imput the missing values by the median method
Bed_med = houses_miss["Tot_Bedrooms"].median()
houses_miss["Tot_Bedrooms"].fillna(Bed_med, inplace=True)

In [ ]:
#We check that we did it well

houses_miss.info()

### 2.1.2 - Case where you have missing values in several variables

Let us now build a dataset with multiple missing values : <br>
Start with a function generating missing values in a chosen column of a dataframe

In [ ]:
def col_miss (df, col, max_miss):
  df = df.copy()
  n_miss = np.random.randint(0, max_miss + 1)
  indices = np.random.choice(df.index, n_miss, replace=False)
  df.loc[indices, col] = np.nan
  return df



In [ ]:
# Copy the train set
housing_miss = houses_df.copy()
housing_miss.info()

Generate some missing values in the first 10 predictors of \<housing_miss>

In [ ]:
cols = housing_miss.columns[:10]

for col in cols:
    housing_miss = col_miss(housing_miss, col, 50)

housing_miss.info()

Now, he have a dataset with missing values in all the quantitative predictors

#### How many missing values are there in each variable ?

In [ ]:
print(housing_miss.isna().sum())

#### Let us use sklearn to do multiple imputation, with existing modules

In [ ]:
# Start with simple imputer
from sklearn.impute import SimpleImputer

**Using Simple Imputer, impute missing data in each variable by replacing missing values with the mean**

In [ ]:
## Using Simple Imputer ##
imputer = SimpleImputer(strategy='mean')

## Imputing missing data in ech variable ##
housing_num = housing_miss.select_dtypes(include=['float64', 'int64'])
X = housing_num.copy()

housing_imputed = pd.DataFrame(
    imputer.fit_transform(housing_num),
    columns=housing_num.columns
)


# Create SimpleImputer
imputer = SimpleImputer(strategy='mean')

# Fit and transform
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print(housing_imputed.isna().sum())

In [ ]:
# Create SimpleImputer
imputer = SimpleImputer(strategy='mean')

# Fit and transform
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
print(housing_imputed.isna().sum())




In [ ]:
X_df = pd.DataFrame(X, columns=df.columns)
X_df.info()

**Using KNNImputer, impute missing data in each variable**

In [ ]:
X_numeric = X_df.copy()  # all numeric predictors

In [ ]:
X_numeric = X_df.select_dtypes(include=['float64', 'int64'])

In [ ]:
from sklearn.impute import KNNImputer
import pandas as pd

X_numeric = X_numeric.dropna(axis=1, how='all')

imputer = KNNImputer(n_neighbors=5)
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_numeric),
    columns=X_numeric.columns
)

print(X_imputed.isna().sum())

## 2.2 - Categorical variables...

In [ ]:
houses_df.info()

### 2.2.1 - Introducing onehot encoding

"Closest_city" has four modalities : the four city names <br>
"income_cat" has five modalities : the five intervals that we have labeled 1,2,3,4,5. However as you see in the graph and in the original values - cat =  [0.4999, 2.3523, 3.1406, 3.9669399999999997, 5.10972, 15.0001] these intervals are not equidistant, so you cannot really add nor substract them meaningfully. <br>
**In short, both categorical variables should be considered nominal**

In [ ]:
# Let us select our categorical variables
houses_cat = houses_df[["Closest_city","income_cat"]]

In [ ]:
# Call for onehot encoder. Choose a dense rather than a sparse vector
from sklearn.preprocessing import OneHotEncoder
onehot = OneHotEncoder(sparse_output=False)
houses_onehot = onehot.fit_transform(houses_cat)

In [ ]:
houses_onehot

In [ ]:
onehot.categories_

### 2.2.2 - Building a Pipeline

Let us first discover how a pipeline operates...

In [ ]:
# Copying once again the original train set
housing_df = train_strat.copy()
housing_df.info()

Let us start with a custom transformer to be used to add attributes

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

# column index
rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room = True): # no *args or **kargs
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self, X, y=None):
        return self  # nothing else to do
    def transform(self, X):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household,
                         bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]


In [ ]:
attr_adder = CombinedAttributesAdder(add_bedrooms_per_room=False)
houses_plus = attr_adder.transform(housing_df.values)

In [ ]:
# Checking the answer...

houses_plus_df = pd.DataFrame(houses_plus,
                              columns=list(housing_df.columns)+["rooms_per_household", "population_per_household"],
                              index=housing_df.index)

houses_plus_df.info()

### 2.2.3 - Pipeline for the quantitative variables

For the quantitative variables, let us generate a pipeline with the following steps
1. Imputing missing values with the "median" method
2. Adding two new attributes : rooms per household and population per household
3. Standardizing the training set

#### Let us restart with a new training set from a dataset with missing values

In [ ]:
train = housing_miss[housing_miss.columns[0:12]]
train.info()

In [ ]:
quanti_features = list(train.columns[:10])
cat_features = ["Closest_city","income_cat"]
train_quanti = train[quanti_features]

In [ ]:
train_quanti.info()

In [ ]:
imputer = SimpleImputer(strategy="median")
train_imputed = imputer.fit_transform(train_quanti)


In [ ]:
train_imputed_df = pd.DataFrame(
    train_imputed,
    columns=train_quanti.columns,
    index=train_quanti.index
)
train_imputed_df.info()

**Defining the pipeline which will go through three steps :**
1. Imputing missing data with the median method
2. Combining three new attributes
3. Standardizing the quantitative features

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

quanti_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('attribs_adder', CombinedAttributesAdder()), # this will add 3 attributes
        ('std_scaler', StandardScaler()),
    ])

houses_quanti = quanti_pipeline.fit_transform(train_imputed_df)

In [ ]:
houses_quanti

### 2.2.4 - Pipeline for the quantitative and categorical variables

Now let us include the categorical variables

In [ ]:
#checking categorical variables at the origin
cat_features = train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Colonnes catégorielles trouvées : {cat_features}")

In [ ]:
#adding categorical variables to the train_imputed_df
train_imputed_df['Closest_city'] = train['Closest_city']
train_imputed_df['income_cat'] = train['income_cat']
train_imputed_df.info()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder as OHE
quanti_features = [col for col in train_imputed_df.columns
                   if col not in ["Median_House_Value", "Closest_city", "income_cat"]]
full_pipeline = ColumnTransformer([
        ("num", quanti_pipeline, quanti_features),
        ("cat", OHE(), cat_features),
    ])

houses_ready = full_pipeline.fit_transform(train_imputed_df)

# Part 3 - Machine Learning

First of all, define the target (outcome) and the predictors (features)

In [ ]:
y = train_imputed_df["Median_House_Value"]
X = full_pipeline.fit_transform(train_imputed_df)


## 3.1 - Learning and evaluating with the training set only

#### Linear Regression
Starting with the most classical Linear Regression <br>
Checking that this algorithm does not use Ordinary Least Square with matrix inversion

In [ ]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X, y)
print("Solver used (not OLS matrix inversion) : SVD via scipy.linalg.lstsq")

Estimating performance with Mean Squared Error and Mean Absolute Error

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
y_pred_lr = lr.predict(X)

In [ ]:
mse = mean_squared_error(y, y_pred_lr)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y, y_pred_lr)

print(f"MSE  : {mse:,.0f}")
print(f"RMSE : {rmse:,.0f}")
print(f"MAE  : {mae:,.0f}")

#### Decision Tree Regression

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=42)
tree.fit(X, y)

y_pred_tree = tree.predict(X)

mse_tree  = mean_squared_error(y, y_pred_tree)
rmse_tree = np.sqrt(mse_tree)
mae_tree  = mean_absolute_error(y, y_pred_tree)

print(f"MSE  : {mse_tree:,.0f}")
print(f"RMSE : {rmse_tree:,.0f}")
print(f"MAE  : {mae_tree:,.0f}")

Estimating performance with Mean Squared Error and Mean Absolute Error

In [ ]:
mse_tree  = mean_squared_error(y, y_pred_tree)
rmse_tree = np.sqrt(mse_tree)
mae_tree  = mean_absolute_error(y, y_pred_tree)

print(f"MSE  : {mse_tree:,.0f}")
print(f"RMSE : {rmse_tree:,.0f}")
print(f"MAE  : {mae_tree:,.0f}")

## 3.2 - Estimating the models with *cross validation*

In [ ]:
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())

#### Linear Regression

In [ ]:
lr = LinearRegression()

In [ ]:
from sklearn.model_selection import cross_val_score

lr_scores = cross_val_score(lr, X, y, scoring="neg_mean_squared_error", cv=10)
lr_rmse = np.sqrt(-lr_scores)
display_scores(lr_rmse)

#### Penalized Linear Regression (Elasticnet)

In [ ]:
from sklearn.linear_model import ElasticNet

In [ ]:
elasticnet = ElasticNet(random_state=42)
elasticnet.fit(X, y)

en_scores = cross_val_score(elasticnet, X, y, scoring="neg_mean_squared_error", cv=10)
en_rmse = np.sqrt(-en_scores)
display_scores(en_rmse)

#### Decision Trees

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=42)

tree_scores = cross_val_score(tree, X, y, scoring="neg_mean_squared_error", cv=10)
tree_rmse = np.sqrt(-tree_scores)
display_scores(tree_rmse)

#### Random Forests

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)

rf_scores = cross_val_score(rf, X, y, scoring="neg_mean_squared_error", cv=10)
rf_rmse = np.sqrt(-rf_scores)
display_scores(rf_rmse)

#### Support Vector Machines

In [ ]:
from sklearn.svm import SVR

svm = SVR()

svm_scores = cross_val_score(svm, X, y, scoring="neg_mean_squared_error", cv=5)
svm_rmse = np.sqrt(-svm_scores)
display_scores(svm_rmse)

## 3.3 - Tuning the model with Grid Search and Randomized Search

#### Example : Random Forest

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
rf_grid = {'n_estimators': [30,60,100], 'max_features': [8,10,15]}

from sklearn.model_selection import GridSearchCV

rf_grid = {'n_estimators': [30,60,100], 'max_features': [8,10,15]}

grid_search_rf = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)
grid_search_rf.fit(X, y)


In [ ]:
## Display the hyperparameters of the best model (code) ##
print("Meilleurs hyperparamètres :", grid_search_rf.best_params_)

In [ ]:
## Print the score of the best model (code)##
best_rmse_rf = np.sqrt(-grid_search_rf.best_score_)
print(f"Meilleur RMSE (CV) : {best_rmse_rf:,.0f}")

#### Example : ElasticNet

In [ ]:
en_grid = {'alpha': np.logspace(-3, 4, 10), 'l1_ratio':np.linspace(0,1,11) }

grid_search_en = GridSearchCV(
    ElasticNet(random_state=42),
    en_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)
grid_search_en.fit(X, y)

In [ ]:
## Display the hyperparameters of the best model (code) ##
print("Meilleurs hyperparamètres :", grid_search_en.best_params_)

In [ ]:
## Print the score of the best model (code)##
best_rmse_en = np.sqrt(-grid_search_en.best_score_)
print(f"Meilleur RMSE (CV) : {best_rmse_en:,.0f}")

#### Example Decision Tree

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
cart_grid = {"min_samples_split": range(1,10),"min_samples_leaf": range(1,60)}

random_search_tree = RandomizedSearchCV(
    DecisionTreeRegressor(random_state=42),
    cart_grid,
    n_iter=20,
    cv=5,
    scoring="neg_mean_squared_error",
    random_state=42,
    n_jobs=-1
)
random_search_tree.fit(X, y)


In [ ]:
## Display the hyperparameters of the best model (code) ##
print("Meilleurs hyperparamètres :", random_search_tree.best_params_)

In [ ]:
## Print the score of the best model (code)##
best_rmse_tree = np.sqrt(-random_search_tree.best_score_)
print(f"Meilleur RMSE (CV) : {best_rmse_tree:,.0f}")

## 3.4 - Final question : how good are our models in predicting unseen data ?

In [ ]:
# Start by checking the structure of the test set (code) #
## Your code here ##
print(test_strat.info())
print(test_strat.describe())
print(test_strat.shape)